In [1]:
# ==========================================
# Step 1: Environment Setup & Dataset Ingestion
# ==========================================

# 1. ติดตั้งไลบรารีเสริม
!pip install -q kagglehub scikit-learn seaborn matplotlib torchsummary

import os
import random
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report
import kagglehub

# 2. ล็อกค่า Random Seed เพื่อให้ผลลัพธ์การทดลองทำซ้ำได้ (Reproducible)
SEED = 101
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# 3. ดาวน์โหลดชุดข้อมูล Cars Body Type Cropped จาก Kaggle
dataset_path = kagglehub.dataset_download("ademboukhris/cars-body-type-cropped")
print(f"Dataset downloaded to: {dataset_path}")

Using device: cuda


100%|██████████| 1.34G/1.34G [00:19<00:00, 75.7MB/s]

Extracting files...


Dataset downloaded to: /root/.cache/kagglehub/datasets/ademboukhris/cars-body-type-cropped/versions/1


In [4]:
# ==========================================
# Step 2: Dataset Exploration & Data Splitting (70/15/15)
# ==========================================

# 1. ค้นหาโฟลเดอร์ย่อย train, valid, test และรวบรวมไฟล์ภาพทั้งหมด
subdirs = [d for d in os.listdir(dataset_path) if os.path.isdir(os.path.join(dataset_path, d))]
base_dir = dataset_path if any(f in subdirs for f in ['train', 'valid', 'test']) else os.path.join(dataset_path, subdirs[0])

all_samples = []
class_names = []

for split_folder in ['train', 'valid', 'test']:
    folder_path = os.path.join(base_dir, split_folder)
    if os.path.exists(folder_path):
        temp_ds = datasets.ImageFolder(root=folder_path)
        all_samples.extend(temp_ds.samples)
        class_names = temp_ds.classes

num_classes = len(class_names)
total_len = len(all_samples)

print(f"Total images found: {total_len:,} images")
print(f"Detected {num_classes} Car Classes: {class_names}")

# 2. แบ่งข้อมูล 70/15/15 ด้วย Seed 101 เพื่อความสามารถในการทำซ้ำ
indices = list(range(total_len))
random.shuffle(indices)

train_split = int(0.70 * total_len)
val_split = int(0.85 * total_len)

train_samples = [all_samples[i] for i in indices[:train_split]]
val_samples = [all_samples[i] for i in indices[train_split:val_split]]
test_samples = [all_samples[i] for i in indices[val_split:]]

print(f"\nData Split Summary (Seed {SEED}):")
print(f"- Train set:      {len(train_samples):,} samples ({len(train_samples)/total_len*100:.1f}%)")
print(f"- Validation set: {len(val_samples):,} samples ({len(val_samples)/total_len*100:.1f}%)")
print(f"- Test set:       {len(test_samples):,} samples ({len(test_samples)/total_len*100:.1f}%)")

# 3. กำหนด Custom Dataset Class สำหรับโหลดภาพและรองรับ Data Transforms
class CarDataset(torch.utils.data.Dataset):
    def __init__(self, samples, transform=None):
        self.samples = samples
        self.transform = transform

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        image = Image.open(path).convert('RGB')
        if self.transform:
            image = self.transform(image)
        return image, label

Total images found: 7,549 images
Detected 7 Car Classes: ['Convertible', 'Coupe', 'Hatchback', 'Pick-Up', 'SUV', 'Sedan', 'VAN']

Data Split Summary (Seed 101):
- Train set:      5,284 samples (70.0%)
- Validation set: 1,132 samples (15.0%)
- Test set:       1,133 samples (15.0%)


In [5]:
# ==========================================
# Step 3: Data Transforms & DataLoaders (Leakage-Free Pipeline)
# ==========================================

IMG_SIZE = 128
BATCH_SIZE = 32

# 1. Pipeline สำหรับชุดฝึกสอน (Train): มี Data Augmentation เสริมความแข็งแกร่ง
train_transforms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),          # รถหันซ้าย/ขวาเป็นประเภทเดิมได้
    transforms.RandomRotation(degrees=10),           # จำลองมุมมองกล้องเอียงเล็กน้อย
    transforms.ColorJitter(brightness=0.2, contrast=0.2), # จำลองสภาพแสงแดดและร่มเงา
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# 2. Pipeline สำหรับชุดตรวจสอบและทดสอบ (Validation/Test): ไม่มี Augmentation
eval_transforms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# 3. ประกอบ Dataset โดยใช้ Transform ที่ถูกต้อง
train_dataset = CarDataset(train_samples, transform=train_transforms)
val_dataset   = CarDataset(val_samples,   transform=eval_transforms)
test_dataset  = CarDataset(test_samples,  transform=eval_transforms)

# 4. สร้าง DataLoader สำหรับป้อนข้อมูลเข้าโมเดล
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

# 5. ทดสอบดึง 1 Batch เพื่อตรวจสอบมิติข้อมูล (Shape Check)
sample_images, sample_labels = next(iter(train_loader))
print("DataLoaders created successfully:")
print(f"- Batch shape: {sample_images.shape} (Batch, Channel, Height, Width)")
print(f"- Label shape: {sample_labels.shape}")
print(f"- Total Batches: Train={len(train_loader)}, Val={len(val_loader)}, Test={len(test_loader)}")

DataLoaders created successfully:
- Batch shape: torch.Size([32, 3, 128, 128]) (Batch, Channel, Height, Width)
- Label shape: torch.Size([32])
- Total Batches: Train=166, Val=36, Test=36


In [6]:
# ==========================================
# Step 4: Model Architectures (Baseline vs. VehicleCNN)
# ==========================================

# 1. โมเดลเปรียบเทียบพื้นฐาน (Single-layer Linear Baseline)
class VehicleBaseline(nn.Module):
    def __init__(self, num_classes=num_classes):
        super(VehicleBaseline, self).__init__()
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(3 * IMG_SIZE * IMG_SIZE, num_classes)
        )

    def forward(self, x):
        return self.classifier(x)

# 2. โมเดลหลักที่นำเสนอ: Custom VehicleCNN (4 Conv Blocks + Regularized Classifier Head)
class VehicleCNN(nn.Module):
    def __init__(self, num_classes=num_classes):
        super(VehicleCNN, self).__init__()

        # Block 1: 3 -> 32 channels (128x128 -> 64x64)
        self.block1 = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2)
        )

        # Block 2: 32 -> 64 channels (64x64 -> 32x32)
        self.block2 = nn.Sequential(
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2)
        )

        # Block 3: 64 -> 128 channels (32x32 -> 16x16)
        self.block3 = nn.Sequential(
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2)
        )

        # Block 4: 128 -> 256 channels (16x16 -> 8x8)
        self.block4 = nn.Sequential(
            nn.Conv2d(128, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2)
        )

        # Classification Head: GAP + Dual Dropout MLP Head
        self.gap = nn.AdaptiveAvgPool2d((1, 1))
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(p=0.3),
            nn.Linear(256, 128),
            nn.ReLU(inplace=True),
            nn.Dropout(p=0.2),
            nn.Linear(128, num_classes)
        )

    def forward(self, x):
        x = self.block1(x)
        x = self.block2(x)
        x = self.block3(x)
        x = self.block4(x)
        x = self.gap(x)
        x = self.classifier(x)
        return x

# 3. ฟังก์ชันนับจำนวน Trainable Parameters
def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

# ประกาศตัวแปรและส่งโมเดลเข้า GPU
baseline_model = VehicleBaseline(num_classes).to(device)
proposed_model = VehicleCNN(num_classes).to(device)

print("=" * 60)
print(f"Baseline Linear Parameters : {count_parameters(baseline_model):,} parameters")
print(f"Custom VehicleCNN Parameters: {count_parameters(proposed_model):,} parameters")
print("=" * 60)

# ทดสอบ Forward Pass ด้วย Dummy Input
dummy_input = torch.randn(2, 3, IMG_SIZE, IMG_SIZE).to(device)
print(f"Input shape       : {dummy_input.shape}")
print(f"Baseline output   : {baseline_model(dummy_input).shape}")
print(f"VehicleCNN output : {proposed_model(dummy_input).shape}")

Baseline Linear Parameters : 344,071 parameters
Custom VehicleCNN Parameters: 423,175 parameters
Input shape       : torch.Size([2, 3, 128, 128])
Baseline output   : torch.Size([2, 7])
VehicleCNN output : torch.Size([2, 7])


In [8]:
# ==========================================
# Step 5: Training & Evaluation Functions
# ==========================================

def train_one_epoch(model, dataloader, criterion, optimizer):
    model.train()
    running_loss, correct, total = 0.0, 0, 0
    for images, labels in dataloader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        _, preds = outputs.max(1)
        correct += preds.eq(labels).sum().item()
        total += labels.size(0)

    epoch_loss = running_loss / total
    epoch_acc = correct / total
    return epoch_loss, epoch_acc

def evaluate(model, dataloader, criterion):
    model.eval()
    running_loss, correct, total = 0.0, 0, 0
    all_preds, all_labels = [], []

    with torch.no_grad():
        for images, labels in dataloader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)

            running_loss += loss.item() * images.size(0)
            _, preds = outputs.max(1)
            correct += preds.eq(labels).sum().item()
            total += labels.size(0)

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    val_loss = running_loss / total
    val_acc = correct / total
    macro_f1 = f1_score(all_labels, all_preds, average='macro', zero_division=0)
    return val_loss, val_acc, macro_f1, all_preds, all_labels

print("Training and Evaluation functions defined successfully.")

Training and Evaluation functions defined successfully.


In [9]:
# ==========================================
# Step 6: Model Training & Test Set Evaluation
# ==========================================

import time

# สร้างโฟลเดอร์สำหรับเก็บผลลัพธ์และ Checkpoint
os.makedirs("outputs", exist_ok=True)

# -------------------------------------------------------------
# 1. ฝึกสอนโมเดลหลัก: Custom VehicleCNN (30 Epochs)
# -------------------------------------------------------------
print("Starting training: Custom VehicleCNN...")
EPOCHS = 30
criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(proposed_model.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-5)

history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}
best_val_loss = float('inf')

start_time = time.time()
for epoch in range(EPOCHS):
    train_loss, train_acc = train_one_epoch(proposed_model, train_loader, criterion, optimizer)
    val_loss, val_acc, val_f1, _, _ = evaluate(proposed_model, val_loader, criterion)
    scheduler.step()

    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['train_acc'].append(train_acc)
    history['val_acc'].append(val_acc)

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(proposed_model.state_dict(), "outputs/best_vehicle_cnn.pth")
        save_indicator = "--> [Saved Best Model]"
    else:
        save_indicator = ""

    print(f"Epoch [{epoch+1:02d}/{EPOCHS}] | Train Loss: {train_loss:.4f} Acc: {train_acc*100:.2f}% | Val Loss: {val_loss:.4f} Acc: {val_acc*100:.2f}% F1: {val_f1:.4f} {save_indicator}")

train_duration = time.time() - start_time
print(f"VehicleCNN training finished in {train_duration/60:.2f} minutes.\n")

# -------------------------------------------------------------
# 2. ฝึกสอนโมเดลเปรียบเทียบ: Baseline Linear Model (10 Epochs)
# -------------------------------------------------------------
print("Starting training: Baseline Linear Model...")
baseline_opt = optim.Adam(baseline_model.parameters(), lr=1e-3)
best_base_loss = float('inf')

for epoch in range(10):
    b_loss, b_acc = train_one_epoch(baseline_model, train_loader, criterion, baseline_opt)
    b_val_loss, b_val_acc, b_val_f1, _, _ = evaluate(baseline_model, val_loader, criterion)
    if b_val_loss < best_base_loss:
        best_base_loss = b_val_loss
        torch.save(baseline_model.state_dict(), "outputs/best_baseline.pth")

print("Baseline training finished.\n")

# -------------------------------------------------------------
# 3. โหลดโมเดลที่ดีที่สุดมาประเมินบน Test Set (1,133 ตัวอย่าง)
# -------------------------------------------------------------
baseline_model.load_state_dict(torch.load("outputs/best_baseline.pth"))
proposed_model.load_state_dict(torch.load("outputs/best_vehicle_cnn.pth"))

_, base_test_acc, base_test_f1, _, _ = evaluate(baseline_model, test_loader, criterion)
_, cnn_test_acc, cnn_test_f1, cnn_preds, cnn_labels = evaluate(proposed_model, test_loader, criterion)

print("=" * 65)
print("              TEST SET EVALUATION SUMMARY (Seed 101)             ")
print("=" * 65)
print(f"1. Baseline Linear   : Accuracy = {base_test_acc*100:.2f}% | Macro-F1 = {base_test_f1:.4f}")
print(f"2. Custom VehicleCNN : Accuracy = {cnn_test_acc*100:.2f}% | Macro-F1 = {cnn_test_f1:.4f}")
print("-" * 65)
print(f"Performance Gain     : Accuracy = +{(cnn_test_acc - base_test_acc)*100:.2f}% | Macro-F1 = +{cnn_test_f1 - base_test_f1:.4f}")
print("=" * 65)

Starting training: Custom VehicleCNN...
Epoch [01/30] | Train Loss: 1.8819 Acc: 22.75% | Val Loss: 1.7852 Acc: 29.15% F1: 0.2503 --> [Saved Best Model]
Epoch [02/30] | Train Loss: 1.7332 Acc: 29.71% | Val Loss: 1.8018 Acc: 28.89% F1: 0.2270 
Epoch [03/30] | Train Loss: 1.6294 Acc: 33.71% | Val Loss: 1.6978 Acc: 32.95% F1: 0.2828 --> [Saved Best Model]
Epoch [04/30] | Train Loss: 1.5314 Acc: 37.59% | Val Loss: 1.6419 Acc: 36.22% F1: 0.3018 --> [Saved Best Model]
Epoch [05/30] | Train Loss: 1.4818 Acc: 39.72% | Val Loss: 1.5852 Acc: 38.07% F1: 0.3300 --> [Saved Best Model]
Epoch [06/30] | Train Loss: 1.4045 Acc: 43.38% | Val Loss: 1.4923 Acc: 37.72% F1: 0.3054 --> [Saved Best Model]
Epoch [07/30] | Train Loss: 1.3652 Acc: 44.38% | Val Loss: 1.3338 Acc: 45.85% F1: 0.4621 --> [Saved Best Model]
Epoch [08/30] | Train Loss: 1.3223 Acc: 46.86% | Val Loss: 1.3531 Acc: 41.43% F1: 0.3504 
Epoch [09/30] | Train Loss: 1.2741 Acc: 49.79% | Val Loss: 1.4892 Acc: 41.78% F1: 0.3759 
Epoch [10/30] | Tr